# SwinV2 Image Classification — DIMER E2E tutorial

**Profile:** `E2E` · **Notebook spec:** 1.0
**Pipeline:** [`kurtvalcorza/swin-classification-pipeline`](https://github.com/kurtvalcorza/swin-classification-pipeline)
**Capability:** image-folder validation → supervised SwinV2 fine-tuning → content-addressed artifact publication → fresh-boundary reload → new-image inference.

## What this notebook does

DIMER's Swin classification pipeline is a *composition* of two pinned workers plus an allowlisted base model:

| Layer | What it supplies | Where it comes from |
|---|---|---|
| Upstream model | SwinV2 Tiny/Small weights pretrained on ImageNet-1k (Microsoft Swin Transformer V2, MIT-licensed weights via `timm`) | Hugging Face Hub, pinned to an immutable revision and SHA-256 in the finetuner's base-model catalog |
| `swin-classification-validate` | Freezes the dataset's logical identity: sample IDs, class labels, `train/`/`val/` ownership, per-file digests | Pinned validator worker release |
| `swin-classification-train` | Supervised **gradient fine-tuning** of the whole network (AdamW, cross-entropy), then artifact publication, fresh reload and evaluation | Pinned finetuner worker release |
| This notebook | Orchestration, tutorial sample/BYOD, baseline, fresh-boundary verification, machine-readable exports | You are here |

**Adaptation type:** full-network gradient fine-tuning. This is not zero-shot inference, in-context conditioning, or preprocessing-only fitting. At inference time the fine-tuned network maps a 256×256 RGB image to one logit per class; the decision rule is `argmax`.

## By the end of this notebook you will be able to

- bootstrap the pipeline and its two workers at the **immutable revisions the release pins**;
- resolve the allowlisted base model and verify the checkpoint bytes by SHA-256 before use;
- prepare a deterministic synthetic sample **or** upload your own image folder (gated, off by default);
- validate the dataset through the real validator worker and read its handoff;
- fine-tune through the real finetuner worker and read its evaluation report;
- compare the model with a majority-class baseline on the frozen validation split;
- reload the exported `safetensors` artifact from a **fresh directory** and prove it reproduces the worker's reported metrics;
- classify a new image and export predictions, metrics and provenance as JSON/CSV.

**This notebook does not demonstrate:** ImageNet benchmark reproduction, calibrated probabilities, CPU training, object detection, semantic segmentation, DIMER serving deployment, or production fitness. Metrics on the default synthetic sample are **sanity evidence only**.

Related: [repository README](https://github.com/kurtvalcorza/swin-classification-pipeline#readme) · [tutorial registry](https://github.com/kurtvalcorza/swin-classification-pipeline/blob/main/tutorials/README.md) · [SwinV2 paper](https://arxiv.org/abs/2111.09883) · [`timm/swinv2_tiny_window8_256.ms_in1k`](https://huggingface.co/timm/swinv2_tiny_window8_256.ms_in1k)


## Prerequisites

- **Runtime:** Python 3.11+ with an NVIDIA GPU exposed as `cuda:0` (Colab **T4** or better; Kaggle T4 also works). The finetuner is **fail-closed**: it refuses to run without the expected accelerator instead of silently falling back to CPU. PyTorch/torchvision come from the runtime image and are reported below; the notebook pins the worker-level dependencies.
- **Knowledge:** basic Python, the `train/<class>/` image-folder convention, and what accuracy / cross-entropy mean.
- **Data:** the default path generates a deterministic synthetic two-class sample (24 PNGs) and needs no private data. A gated BYOD path accepts a ZIP of your own image folder (schema in Section 3).
- **External access:** GitHub (clone the pipeline and workers) and the Hugging Face Hub (download the pinned checkpoint). No other service is contacted; uploaded data stays inside this runtime.
- **Credentials:** the pinned validator and finetuner repositories are **private**. Provide a Colab Secret named `GITHUB_TOKEN` (or a `GITHUB_TOKEN` environment variable / Kaggle secret elsewhere) with *read* access to `kurtvalcorza/swin-classification-dataset-validator` and `kurtvalcorza/swin-classification-finetuner`. The token is passed to Git through an ephemeral HTTP header — it is never printed, embedded in a URL, written to Git config, or exported.
- **Time and memory:** on the default sample the whole notebook is dominated by installs and the 113 MB checkpoint download; training one epoch on 16 images takes seconds. These are estimates until the Kaggle T4 execution record in `tutorials/RELEASE_VERIFICATION.md` is consulted; the finetuner's formal qualification packet measured ~1.7 GiB peak reserved VRAM for SwinV2-Tiny at batch 8 on an RTX 5070 Ti, which is not a Colab measurement.

> **Do not upload confidential or restricted images** to a runtime you are not authorised to use for that data.


## 1. Bootstrap immutable sources and the pinned runtime

Everything downstream is anchored to `PIPELINE_REF`, a specific commit of the pipeline repository. From that commit the notebook reads `pipeline-manifest.json` and the two `release/worker-release-*.json` files, which name the **exact validator and finetuner commits** the release was composed from. The notebook never chooses a worker revision itself.

Dependencies are pinned to the versions the finetuner declares (`timm==1.0.28`, `Pillow==12.3.0`) plus pinned `huggingface_hub`/`safetensors`; the two workers are then installed from their pinned checkouts with `--no-deps`, so only one dependency graph is resolved. PyTorch is *not* reinstalled — the runtime's build is used and reported.

**Look for:** a JSON block naming the three source revisions and the runtime (Python, torch, torchvision, timm, CUDA, GPU). If the GPU line is `null`, switch to a GPU runtime and run again.


In [ ]:
from __future__ import annotations

import base64
import csv
import hashlib
import json
import os
import platform
import random
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path, PurePosixPath

# --- Immutable anchors -----------------------------------------------------------------------
PIPELINE_REPO = "https://github.com/kurtvalcorza/swin-classification-pipeline.git"
PIPELINE_REF = "e73a8aee9881093861bbd934b8476626fad23f12"
VALIDATOR_REPO = "https://github.com/kurtvalcorza/swin-classification-dataset-validator.git"
FINETUNER_REPO = "https://github.com/kurtvalcorza/swin-classification-finetuner.git"

WORK = Path("/content/dimer-swin-classification")
PIPELINE_DIR, VALIDATOR_DIR, FINETUNER_DIR = WORK / "pipeline", WORK / "validator", WORK / "finetuner"
WORK.mkdir(parents=True, exist_ok=True)

PINNED = {"timm": "1.0.28", "Pillow": "12.3.0", "huggingface_hub": "1.29.0", "safetensors": "0.8.0"}


def run(command, cwd=None, env=None):
    """Run a subprocess, echo the command, and raise on non-zero exit."""
    command = [str(part) for part in command]
    print("+", " ".join(command))
    subprocess.run(command, cwd=cwd, check=True, env=env)


def resolve_github_token():
    """Return a read token for the private worker repositories, or raise a clear error.

    Resolution order: GITHUB_TOKEN environment variable, Colab Secrets, Kaggle Secrets.
    The value is only ever handed to Git as an HTTP header (see private_git_env)."""
    token = os.environ.get("GITHUB_TOKEN", "").strip()
    if not token:
        try:  # Colab
            from google.colab import userdata
            token = (userdata.get("GITHUB_TOKEN") or "").strip()
        except Exception:
            token = ""
    if not token:
        try:  # Kaggle (Add-ons > Secrets, attached to the notebook)
            from kaggle_secrets import UserSecretsClient
            token = (UserSecretsClient().get_secret("GITHUB_TOKEN") or "").strip()
        except Exception:
            token = ""
    if not token:
        raise RuntimeError(
            "The validator and finetuner repositories are private. Add a GITHUB_TOKEN secret "
            "(Colab: key icon in the left sidebar; Kaggle: Add-ons > Secrets) with read access to "
            "kurtvalcorza/swin-classification-dataset-validator and kurtvalcorza/swin-classification-finetuner, "
            "then run this cell again."
        )
    return token


def private_git_env(token):
    """Git environment that authenticates via an ephemeral extraHeader (never a URL, never config on disk)."""
    credential = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = os.environ.copy()
    env.update({
        "GIT_TERMINAL_PROMPT": "0",
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraHeader",
        "GIT_CONFIG_VALUE_0": f"Authorization: Basic {credential}",
    })
    return env


def checkout_pinned(url, ref, dst, env=None):
    """Clone (blobless) if needed, then detach at the exact pinned commit."""
    if not dst.exists():
        run(["git", "clone", "--filter=blob:none", url, dst], env=env)
    run(["git", "checkout", "--detach", ref], cwd=dst, env=env)
    head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=dst, check=True, capture_output=True, text=True).stdout.strip()
    if head != ref:
        raise RuntimeError(f"{dst.name}: checked out {head}, expected {ref}")


# --- Public pipeline repository at the immutable anchor --------------------------------------
checkout_pinned(PIPELINE_REPO, PIPELINE_REF, PIPELINE_DIR)
pipeline_manifest = json.loads((PIPELINE_DIR / "pipeline-manifest.json").read_text())
validator_release = json.loads((PIPELINE_DIR / "release/worker-release-validator.json").read_text())
finetuner_release = json.loads((PIPELINE_DIR / "release/worker-release-finetuner.json").read_text())
VALIDATOR_REF = validator_release["sourceRevision"]
FINETUNER_REF = finetuner_release["sourceRevision"]

# --- Private worker repositories at the revisions the release pins ---------------------------
PRIVATE_TOKEN = resolve_github_token()
PRIVATE_GIT_ENV = private_git_env(PRIVATE_TOKEN)
checkout_pinned(VALIDATOR_REPO, VALIDATOR_REF, VALIDATOR_DIR, env=PRIVATE_GIT_ENV)
checkout_pinned(FINETUNER_REPO, FINETUNER_REF, FINETUNER_DIR, env=PRIVATE_GIT_ENV)
del PRIVATE_TOKEN, PRIVATE_GIT_ENV

# --- One dependency graph: pinned worker-level packages, then the workers with --no-deps -------
run([sys.executable, "-m", "pip", "install", "-q", "timm==1.0.28", "huggingface_hub==1.29.0", "safetensors==0.8.0", "pillow==12.3.0"])
run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", VALIDATOR_DIR])
run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", FINETUNER_DIR])
import importlib
importlib.invalidate_caches()  # the worker packages were installed after this kernel started

import PIL
import timm
import torch
import torchvision
from PIL import Image, ImageDraw

# Restart-boundary check (a package imported before the install above keeps its old version in this
# kernel until the runtime restarts). timm builds the model in-kernel, so it must match exactly.
# Pillow only draws/decodes tutorial images in-kernel — the worker CLIs run in fresh processes with the
# pinned 12.3.0 — so a mismatch is reported and recorded rather than fatal.
if timm.__version__ != PINNED["timm"]:
    raise RuntimeError(f"timm {timm.__version__} is active but {PINNED['timm']} was installed. Restart the runtime (Runtime > Restart session) and run from the top.")
if PIL.__version__ != PINNED["Pillow"]:
    print(f"WARNING: this kernel had Pillow {PIL.__version__} imported before the pinned {PINNED['Pillow']} was installed. "
          "The validator/finetuner subprocesses use the pinned version; restart the runtime for exact in-kernel parity.")


def worker_cli(name):
    """Locate an installed worker console script; prefer the interpreter's own bin/ directory."""
    candidate = Path(sys.executable).parent / name
    if candidate.is_file():
        return str(candidate)
    found = shutil.which(name)
    if found:
        return found
    raise RuntimeError(f"{name} was not installed into this runtime; rerun Section 1 after a runtime restart.")

runtime = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "timm": timm.__version__,
    "pillowInKernel": PIL.__version__,
    "pillowPinnedForWorkers": PINNED["Pillow"],
    "cuda": torch.version.cuda,
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
print(json.dumps({
    "sources": {"pipeline": PIPELINE_REF, "validator": VALIDATOR_REF, "finetuner": FINETUNER_REF},
    "runtime": runtime,
}, indent=2))

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required by the fail-closed finetuner. In Colab choose Runtime > Change runtime type > GPU, then Run all.")


## 2. Resolve the pinned base model

The finetuner ships a **base-model catalog**: an allowlist mapping each catalog key to a Hugging Face repository, an immutable commit revision, the exact file, its SHA-256, licence, and qualification status. Off-catalog models are refused and the worker never downloads weights itself — the notebook stages the file and the worker verifies the digest again before loading.

Two entries are supported; the form below selects one. The pipeline's own `provenance/open-weights.json` records the same revision and digest, and the cell asserts the two agree so a drift in either file is caught here rather than during training.

**Look for:** the model identity block with `revision` (a 40-hex Hugging Face commit) and `digest` (`sha256:…`). No weights are downloaded yet.


In [ ]:
MODEL_KEY = "swinv2-tiny-window8-256-ms-in1k"  # @param ["swinv2-tiny-window8-256-ms-in1k", "swinv2-small-window8-256-ms-in1k"]

catalog = json.loads((FINETUNER_DIR / "catalog/base-model-catalog.json").read_text())
if MODEL_KEY not in catalog["entries"]:
    raise RuntimeError(f"{MODEL_KEY!r} is not an allowlisted catalog entry: {sorted(catalog['entries'])}")
catalog_entry = catalog["entries"][MODEL_KEY]
weight_file = catalog_entry["source"]["files"][0]

provenance = json.loads((PIPELINE_DIR / "provenance/open-weights.json").read_text())
pipeline_entry = next(x for x in provenance["operationalWeights"] if x["catalogKey"] == MODEL_KEY)
assert catalog_entry["source"]["revision"] == pipeline_entry["modelDescriptorVersion"], "catalog/provenance revision drift"
assert weight_file["digest"] == "sha256:" + pipeline_entry["sha256"], "catalog/provenance digest drift"
assert catalog_entry["qualification"]["status"] == "QUALIFIED", catalog_entry["qualification"]

model_identity = {
    "modelKey": MODEL_KEY,
    "modelDescriptorId": catalog_entry["modelDescriptorId"],
    "timmModelName": catalog_entry["timmModelName"],
    "repo": catalog_entry["source"]["repoId"],
    "revision": catalog_entry["source"]["revision"],
    "file": weight_file["path"],
    "digest": weight_file["digest"],
    "license": catalog_entry.get("license"),
}
print(json.dumps(model_identity, indent=2))


## 3. Prepare the dataset: default synthetic sample or bring your own

### Expected input schema (image-folder representation)

```
<dataset root>/
  train/<class name>/*.png|jpg|jpeg|bmp|tif|tiff|webp
  val/<class name>/...          (or valid/ — exactly one of the two)
  test/<class name>/...         (optional; never reinterpreted as validation)
```

Class names are directory names and are **case-sensitive**. Every class should appear in both `train/` and `val/` (the validator warns otherwise, and the evaluation would be meaningless). Files with unrecognised extensions and undecodable images are **fatal** validator findings — nothing is silently skipped or substituted. There is no image-count ceiling in the validator; GPU memory and the batch size are the practical limits.

### Default path (runs without any upload)

`USE_BYOD_DATASET = False` generates 24 deterministic 256×256 PNGs from a fixed seed: class `cool` (blue background, white square) and class `warm` (red background, cream circle), 8 training and 4 validation images per class. This is **smoke data**: it proves the plumbing and cannot say anything about real-world accuracy.

### BYOD path (gated)

Set `USE_BYOD_DATASET = True` and run the cell; in Colab an upload dialog asks for **one ZIP** laid out as above (a single top-level folder inside the ZIP is fine). The archive is extracted with path-safety checks — absolute paths, `..`, backslash-ambiguous names, symlinks, anything escaping the extraction root, or an expanded size above `MAX_EXPANDED_MIB` are rejected. Your images stay in this runtime and are only read by the local validator and finetuner. Outside Colab, set `BYOD_ZIP_PATH` instead of using the dialog.

**Look for:** a small table of images per split and class. Every class should have at least one image in both `train` and `val`.


In [ ]:
USE_BYOD_DATASET = False  # @param {type:"boolean"}
BYOD_ZIP_PATH = ""  # @param {type:"string"}
MAX_EXPANDED_MIB = 2048  # @param {type:"integer"}
SAMPLE_SEED = 20260910  # @param {type:"integer"}

DATASET_DIR = WORK / "dataset"
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


def safe_extract_zip(zip_path: Path, destination: Path, max_expanded_bytes: int) -> int:
    """Extract an image-folder ZIP with the archive-safety rules of DIMER Notebook Spec section 19."""
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    expanded = 0
    with zipfile.ZipFile(zip_path) as archive:
        for info in archive.infolist():
            name = info.filename
            posix = PurePosixPath(name)
            if "\\" in name:
                raise ValueError(f"backslash-ambiguous archive member rejected: {name!r}")
            if posix.is_absolute() or name.startswith("/") or ".." in posix.parts:
                raise ValueError(f"absolute or traversing archive member rejected: {name!r}")
            if (info.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError(f"symlink archive member rejected: {name!r}")
            expanded += info.file_size
            if expanded > max_expanded_bytes:
                raise ValueError(f"archive expands beyond {max_expanded_bytes} bytes; raise MAX_EXPANDED_MIB only if you trust the file")
            target = (root / posix).resolve()
            if root != target and root not in target.parents:
                raise ValueError(f"archive member escapes extraction root: {name!r}")
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as src, target.open("wb") as dst:
                shutil.copyfileobj(src, dst)
    return expanded


def locate_dataset_root(extracted: Path) -> Path:
    """Accept either `train/` at the top level or exactly one wrapper directory around it."""
    if (extracted / "train").is_dir():
        return extracted
    children = [p for p in extracted.iterdir() if p.is_dir() and not p.name.startswith(("__MACOSX", "."))]
    if len(children) == 1 and (children[0] / "train").is_dir():
        return children[0]
    raise ValueError("ZIP must contain train/ (and val/ or valid/) at the top level or inside one wrapper folder")


if USE_BYOD_DATASET:
    if BYOD_ZIP_PATH:
        zip_path = Path(BYOD_ZIP_PATH)
    else:
        try:
            from google.colab import files
        except ImportError as error:
            raise RuntimeError("Not running in Colab: set BYOD_ZIP_PATH to a ZIP already present in this runtime.") from error
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise RuntimeError("Upload exactly one ZIP file.")
        zip_path = WORK / next(iter(uploaded))
        zip_path.write_bytes(next(iter(uploaded.values())))
    extracted = WORK / "byod-extracted"
    if extracted.exists():
        shutil.rmtree(extracted)
    expanded = safe_extract_zip(zip_path, extracted, MAX_EXPANDED_MIB * 1024 * 1024)
    shutil.copytree(locate_dataset_root(extracted), DATASET_DIR)
    dataset_origin = {"type": "user-supplied image folder (BYOD)", "zip": zip_path.name, "expandedBytes": expanded}
else:
    rng = random.Random(SAMPLE_SEED)
    for split, per_class in {"train": 8, "val": 4}.items():
        for class_name in ("cool", "warm"):
            out = DATASET_DIR / split / class_name
            out.mkdir(parents=True, exist_ok=True)
            for index in range(per_class):
                background = (35, 70, 190) if class_name == "cool" else (190, 65, 35)
                image = Image.new("RGB", (256, 256), background)
                draw = ImageDraw.Draw(image)
                jitter = rng.randint(-15, 15)
                if class_name == "cool":
                    draw.rectangle((60 + jitter, 60, 196 + jitter, 196), outline=(230, 240, 255), width=10)
                else:
                    draw.ellipse((60 + jitter, 60, 196 + jitter, 196), outline=(255, 240, 220), width=10)
                image.save(out / f"{class_name}-{index:02d}.png")
    dataset_origin = {"type": "deterministic synthetic tutorial sample", "seed": SAMPLE_SEED, "generator": "this notebook, Section 3"}

# Compact inventory so you can sanity-check the layout before validation.
inventory = {}
for path in sorted(DATASET_DIR.rglob("*")):
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
        split, class_name = path.relative_to(DATASET_DIR).parts[:2]
        inventory.setdefault(class_name, {}).setdefault(split, 0)
        inventory[class_name][split] += 1
splits = sorted({s for counts in inventory.values() for s in counts})
print(f"{'class':<24}" + "".join(f"{s:>8}" for s in splits))
for class_name, counts in inventory.items():
    print(f"{class_name:<24}" + "".join(f"{counts.get(s, 0):>8}" for s in splits))
print(json.dumps(dataset_origin, indent=2))


## 4. Validate through the real dataset worker

`swin-classification-validate` is the pinned validator CLI. It decodes every image (decode failure is fatal), maps `train/` → `train`, `val/`|`valid/` → `validation`, `test/` → `test` without ever reinterpreting a split, assigns a stable sample ID (the relative path) to every file, digests every file, and writes a **handoff** of four JSON documents: `logical-dataset-manifest.json`, `data-plan.json`, `semantic-dataset-schema.json` and `validated-dataset-manifest.json`. The finetuner consumes that handoff *as is* — it cannot re-split, drop or add samples.

In DIMER the four `--*-digest` arguments bind the run to the platform's job spec, admission record and security grant. In this tutorial they are deterministic stand-ins derived from fixed content, so the run is reproducible but **not** a production DIMER admission.

**Look for:** `"state": "SUCCEEDED"`, the frozen `labelMap` (class index → class name), split counts, and any `WARNING` findings (for example a class missing from a required split).


In [ ]:
def digest_json(document) -> str:
    """Canonical JSON SHA-256 in the `sha256:<hex>` form the workers use."""
    encoded = json.dumps(document, sort_keys=True, separators=(",", ":")).encode()
    return "sha256:" + hashlib.sha256(encoded).hexdigest()


binding = {
    "job": digest_json({"kind": "tutorial-job", "task": pipeline_manifest["taskProfile"]}),
    "admission": digest_json({"kind": "tutorial-admission"}),
    "security": digest_json({"kind": "tutorial-security", "networkDuringRunning": "DENY"}),
}
HANDOFF_DIR = WORK / "validated"
if HANDOFF_DIR.exists():
    shutil.rmtree(HANDOFF_DIR)

run([
    worker_cli("swin-classification-validate"), DATASET_DIR, HANDOFF_DIR,
    "--job-id", "tutorial-classification", "--attempt-id", "attempt-1",
    "--worker-release-digest", pipeline_manifest["validatorWorkerReleaseDigest"],
    "--effective-job-spec-digest", binding["job"],
    "--admission-record-digest", binding["admission"],
    "--security-grant-digest", binding["security"],
])

validation_result = json.loads((HANDOFF_DIR / "result.json").read_text())
if validation_result.get("state") != "SUCCEEDED":
    findings_path = HANDOFF_DIR / "validation-findings.json"
    detail = findings_path.read_text() if findings_path.exists() else json.dumps(validation_result, indent=2)
    raise RuntimeError("Dataset validation failed. Fix the dataset layout and rerun Section 3 and 4.\n" + detail)

semantic_schema = json.loads((HANDOFF_DIR / "semantic-dataset-schema.json").read_text())
data_plan = json.loads((HANDOFF_DIR / "data-plan.json").read_text())
logical_manifest = json.loads((HANDOFF_DIR / "logical-dataset-manifest.json").read_text())
label_map = semantic_schema["labelMap"]
class_names = [label_map[key] for key in sorted(label_map, key=int)]  # index -> class name, frozen by the validator
split_counts = {}
for assignment in data_plan["assignments"]:
    split_counts[assignment["split"]] = split_counts.get(assignment["split"], 0) + 1
findings_path = HANDOFF_DIR / "validation-findings.json"
warnings = json.loads(findings_path.read_text()).get("findings", []) if findings_path.exists() else []

print(json.dumps({
    "state": validation_result["state"],
    "logicalDatasetDigest": logical_manifest["logicalDatasetDigest"],
    "labelMap": label_map,
    "splitCounts": split_counts,
    "warnings": [{"code": w.get("code"), "detail": w.get("observed")} for w in warnings],
}, indent=2))
if split_counts.get("validation", 0) == 0:
    raise RuntimeError("No validation samples were assigned; the evaluation below would be empty.")


## 5. Stage and SHA-256 verify the pinned checkpoint

The checkpoint is downloaded from the Hugging Face repository **at the immutable revision** recorded in the catalog, copied into the layout the finetuner expects (`<weights root>/<model key>/<file>`), and hashed. A digest mismatch deletes the file and stops the notebook — the worker would refuse it anyway, but failing here gives a clearer message.

The file is `model.safetensors`: a tensor-only format with no code execution on load, so no `trust_remote_code` or pickle trust decision is needed anywhere in this pipeline.

Digest equality proves the bytes are the ones the catalog allowlisted; it does **not** by itself prove who published them — that trust rests on the catalog review that produced the pin.

**Look for:** `expected` and `observed` digests that are identical.


In [ ]:
from huggingface_hub import hf_hub_download

WEIGHTS_DIR = WORK / "weights"
staged_file = WEIGHTS_DIR / MODEL_KEY / weight_file["path"]
staged_file.parent.mkdir(parents=True, exist_ok=True)

downloaded = Path(hf_hub_download(
    repo_id=catalog_entry["source"]["repoId"],
    filename=weight_file["path"],
    revision=catalog_entry["source"]["revision"],
))
shutil.copyfile(downloaded, staged_file)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return "sha256:" + digest.hexdigest()


observed_digest = sha256_file(staged_file)
print(json.dumps({"file": str(staged_file), "bytes": staged_file.stat().st_size,
                  "expected": weight_file["digest"], "observed": observed_digest}, indent=2))
if observed_digest != weight_file["digest"]:
    staged_file.unlink(missing_ok=True)
    raise RuntimeError("Checkpoint digest mismatch: the downloaded bytes are not the allowlisted checkpoint. Do not proceed.")


## 6. Fine-tune through the real finetuner worker

`swin-classification-train` is the pinned production-facing CLI. Given the dataset, the validator handoff, the staged weights and the catalog it:

1. re-verifies the handoff digests and the staged checkpoint digest (refusing on any mismatch);
2. requires `--expected-accelerator cuda:0` to match the observed device — **no silent CPU fallback**;
3. builds the timm SwinV2 model from the staged file, replaces the ImageNet head with one output per class, and fine-tunes **all parameters** with AdamW and cross-entropy (256×256 bicubic resize, ImageNet mean/std normalisation, no augmentation);
4. publishes a content-addressed artifact generation (`model.safetensors` + `model-config.json` + `artifact-manifest.json` with per-member digests);
5. **frees the trained model**, reloads the persisted artifact, and evaluates the frozen `validation` split with it — the reported metrics come from the persisted bytes, not the in-memory model.

Seeds control data order and initialisation of the new head. The run is *re-executable*, not bitwise reproducible: cuDNN kernel selection and floating-point reduction order on the GPU still vary, so expect identical accuracy on this sample but small drift in loss and a different artifact digest between runs.

**Look for:** `"state": "SUCCEEDED"`, the per-epoch `history`, and the two evaluation metrics. This is the slow cell (seconds on the sample; minutes on a real dataset).


In [ ]:
EPOCHS = 1  # @param {type:"integer"}
BATCH_SIZE = 4  # @param {type:"integer"}
LEARNING_RATE = 1e-4  # @param {type:"number"}
SEED = 20260910  # @param {type:"integer"}

TRAINING_DIR = WORK / "training-output"
if TRAINING_DIR.exists():
    shutil.rmtree(TRAINING_DIR)

run([
    worker_cli("swin-classification-train"), DATASET_DIR, HANDOFF_DIR, WEIGHTS_DIR, FINETUNER_DIR / "catalog/base-model-catalog.json", TRAINING_DIR,
    "--model-key", MODEL_KEY, "--expected-accelerator", "cuda:0",
    "--job-id", "tutorial-classification", "--attempt-id", "attempt-1",
    "--worker-release-digest", pipeline_manifest["finetunerWorkerReleaseDigest"],
    "--effective-job-spec-digest", binding["job"],
    "--admission-record-digest", binding["admission"],
    "--security-grant-digest", binding["security"],
    "--epochs", EPOCHS, "--batch-size", BATCH_SIZE, "--learning-rate", LEARNING_RATE, "--seed", SEED,
])

training_result = json.loads((TRAINING_DIR / "result.json").read_text())
if training_result.get("state") != "SUCCEEDED":
    raise RuntimeError("Fine-tuning did not succeed; inspect training-output/result.json and run-manifest.json.\n" + json.dumps(training_result, indent=2))
run_manifest = json.loads((TRAINING_DIR / "run-manifest.json").read_text())
evaluation_report = json.loads((TRAINING_DIR / "evaluation-report.json").read_text())
reported_metrics = {metric["id"]: metric["value"] for metric in evaluation_report["metrics"]}

print(json.dumps({
    "state": training_result["state"],
    "device": run_manifest["observed"]["device"],
    "splitCounts": run_manifest["observed"]["splitCounts"],
    "history": run_manifest["observed"]["history"],
    "reportedMetrics": reported_metrics,
    "artifactBundleDigest": run_manifest["observed"]["artifactBundleDigest"],
    "reproducibility": run_manifest["reproducibility"],
}, indent=2))


## 7. Read the evaluation against a trivial baseline

Two metrics are reported by the worker, both on the frozen `validation` split as a **single holdout** — no cross-validation, no repeated runs, therefore no dispersion estimate:

- `core.metric.classification.accuracy` — fraction of validation images whose `argmax` class equals the label. Easy to read, blind to *how* confident the wrong answers were.
- `org.valcorza.metric.classification.cross-entropy` — mean negative log-likelihood of the true class. Sensitive to the score assigned to the correct class, so it moves before accuracy does on tiny datasets.

The **majority-class baseline** is the accuracy you would get by always predicting the most frequent validation class. It is computed from the validator's data plan, so it is correct for BYOD datasets too. On the default balanced sample it is 0.5.

On the synthetic sample these are **tutorial metrics**: the two classes differ in colour and shape, so any working pipeline reaches 1.0 quickly. Treat them as evidence that the pipeline works, not that the model is good.


In [ ]:
validation_sample_ids = sorted(a["sampleId"] for a in data_plan["assignments"] if a["split"] == "validation")
validation_labels = [Path(sample_id).parts[1] for sample_id in validation_sample_ids]  # class = 2nd path component
label_counts = {name: validation_labels.count(name) for name in class_names}
majority_class = max(label_counts, key=label_counts.get)
majority_baseline_accuracy = label_counts[majority_class] / len(validation_labels)

comparison = {
    "estimationProcedure": "single frozen validation holdout (validator-assigned); no dispersion estimate",
    "validationSamples": len(validation_labels),
    "validationLabelCounts": label_counts,
    "majorityClass": majority_class,
    "majorityBaselineAccuracy": majority_baseline_accuracy,
    "modelAccuracy": reported_metrics["core.metric.classification.accuracy"],
    "modelCrossEntropy": reported_metrics["org.valcorza.metric.classification.cross-entropy"],
    "evidenceClass": "tutorial/sanity metrics" if not USE_BYOD_DATASET else "single-holdout metrics on user data",
}
print(json.dumps(comparison, indent=2))
if comparison["modelAccuracy"] < majority_baseline_accuracy:
    print("WARNING: the fine-tuned model does not beat the majority baseline on this holdout. More epochs or more data are needed before drawing any conclusion.")


## 8. Fresh-boundary verification of the exported artifact

A working in-memory model proves nothing about the bytes on disk. This section reproduces what a downstream consumer would do with the artifact and nothing else:

1. copy the published generation directory to a **fresh location** (`WORK/fresh-reload/`) so no path from training is reused;
2. read `artifact-manifest.json` and re-verify every member's SHA-256 (a missing or altered file fails here);
3. rebuild the network from `model-config.json` (`timmModelName`, `numClasses`, `classNames`) and load `model.safetensors` with `strict=True`;
4. run the frozen validation split through it using the finetuner's **own** image decoding (`load_visual_image`, which applies EXIF orientation) and normalisation constants, imported from the installed worker rather than retyped;
5. compare with the worker's evaluation report: accuracy must match **exactly** (deterministic `argmax` on identical inputs), cross-entropy within `1e-4`.

Passing step 3 means *"artifact reloaded"*; passing step 5 means *"artifact reproduces the reported outputs"* — the stronger claim this section exists to make. A per-sample prediction table is kept for export.

The artifact contains only weights and configuration — no training images — but it was **derived from** your dataset; apply your data's licence and confidentiality rules to it.


In [ ]:
import torch.nn.functional as F
from safetensors.torch import load_file
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from swin_classification_finetuner.trainer import NORMALIZATION_MEAN, NORMALIZATION_STD, load_visual_image

# 1. Copy the published generation to a fresh directory.
current_generation = (TRAINING_DIR / "artifact/CURRENT").read_text().strip()
FRESH_DIR = WORK / "fresh-reload"
if FRESH_DIR.exists():
    shutil.rmtree(FRESH_DIR)
shutil.copytree(TRAINING_DIR / "artifact/generations" / current_generation, FRESH_DIR)

# 2. Re-verify every manifest member from the copy.
artifact_manifest = json.loads((FRESH_DIR / "artifact-manifest.json").read_text())


def member_path_of(member) -> str:
    """Each manifest member names its file through a `member-path` relationship."""
    return next(r["path"] for r in member["relationships"] if r["type"] == "org.valcorza.bundle.member-path")


verified_members = []
for member in artifact_manifest["members"]:
    member_file = FRESH_DIR / member_path_of(member)
    if not member_file.is_file():
        raise RuntimeError(f"artifact member missing after copy: {member_file.name} ({member['role']})")
    if sha256_file(member_file) != member["digest"]:
        raise RuntimeError(f"artifact member digest mismatch: {member_file.name} ({member['role']})")
    verified_members.append({"role": member["role"], "file": member_file.name, "digest": member["digest"]})
print("artifact members verified:", json.dumps(verified_members, indent=2))

# 3. Rebuild the model from the artifact contract alone.
model_config = json.loads((FRESH_DIR / "model-config.json").read_text())
assert model_config["classNames"] == class_names, "artifact class order differs from the validator label map"
DEVICE = torch.device("cuda:0")
reloaded_model = timm.create_model(model_config["timmModelName"], pretrained=False, num_classes=model_config["numClasses"])
reloaded_model.load_state_dict(load_file(FRESH_DIR / "model.safetensors"), strict=True)
reloaded_model = reloaded_model.to(DEVICE).eval()

# 4. Score the frozen validation split with the worker's own preprocessing semantics.
size = (model_config["input"]["height"], model_config["input"]["width"])
preprocess = transforms.Compose([
    transforms.Resize(size, interpolation=InterpolationMode.BICUBIC, antialias=True),
    transforms.ToTensor(),
    transforms.Normalize(NORMALIZATION_MEAN, NORMALIZATION_STD),
])


def classify(image_paths):
    """Return (predicted index, softmax scores) per image using the reloaded model and argmax rule."""
    batch = torch.stack([preprocess(load_visual_image(Path(p))) for p in image_paths]).to(DEVICE)
    with torch.no_grad():
        logits = reloaded_model(batch)
    return logits.argmax(dim=1).cpu(), F.softmax(logits, dim=1).cpu(), logits.cpu()


per_sample = []
correct, loss_sum = 0, 0.0
for start in range(0, len(validation_sample_ids), 16):
    ids = validation_sample_ids[start:start + 16]
    predicted, scores, logits = classify([DATASET_DIR / sample_id for sample_id in ids])
    targets = torch.tensor([class_names.index(Path(sample_id).parts[1]) for sample_id in ids])
    loss_sum += float(F.cross_entropy(logits, targets, reduction="sum"))
    correct += int((predicted == targets).sum())
    for i, sample_id in enumerate(ids):
        per_sample.append({"sampleId": sample_id, "label": class_names[int(targets[i])], "predicted": class_names[int(predicted[i])],
                           **{f"score_{name}": float(scores[i][k]) for k, name in enumerate(class_names)}})

# 5. Compare with the worker's report.
reload_check = {
    "reloadedAccuracy": correct / len(validation_sample_ids),
    "reportedAccuracy": reported_metrics["core.metric.classification.accuracy"],
    "reloadedCrossEntropy": loss_sum / len(validation_sample_ids),
    "reportedCrossEntropy": reported_metrics["org.valcorza.metric.classification.cross-entropy"],
    "crossEntropyTolerance": 1e-4,
}
reload_check["accuracyMatches"] = reload_check["reloadedAccuracy"] == reload_check["reportedAccuracy"]
reload_check["crossEntropyMatches"] = abs(reload_check["reloadedCrossEntropy"] - reload_check["reportedCrossEntropy"]) <= reload_check["crossEntropyTolerance"]
print(json.dumps(reload_check, indent=2))
if not (reload_check["accuracyMatches"] and reload_check["crossEntropyMatches"]):
    raise RuntimeError("The reloaded artifact does not reproduce the worker's reported metrics. Do not ship this artifact.")
print(f"Fresh-boundary verification PASSED: artifact reproduces the reported metrics on {len(per_sample)} validation samples.")


## 9. Classify a new image

Real use means images the model has never seen. The default is a freshly drawn `warm`-style image that is in neither split; set `USE_BYOD_IMAGE = True` to upload one of your own instead (any of the accepted extensions; EXIF orientation is honoured exactly as during training).

The decision rule is `argmax(logits)`. The softmax scores are **uncalibrated class scores** — larger means the model favours that class, but 0.9 does not mean "90 % chance of being right". The pipeline ships no confidence threshold; if a downstream system needs one, calibrate it on labelled data from your deployment domain.

**Look for:** `predictedClass` and the per-class scores, which sum to 1.


In [ ]:
USE_BYOD_IMAGE = False  # @param {type:"boolean"}
BYOD_IMAGE_PATH = ""  # @param {type:"string"}

NEW_IMAGE_DIR = WORK / "new-images"
NEW_IMAGE_DIR.mkdir(exist_ok=True)
if USE_BYOD_IMAGE:
    if BYOD_IMAGE_PATH:
        new_image_path = Path(BYOD_IMAGE_PATH)
    else:
        try:
            from google.colab import files
        except ImportError as error:
            raise RuntimeError("Not running in Colab: set BYOD_IMAGE_PATH to an image already present in this runtime.") from error
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise RuntimeError("Upload exactly one image.")
        new_image_path = NEW_IMAGE_DIR / next(iter(uploaded))
        new_image_path.write_bytes(next(iter(uploaded.values())))
    if new_image_path.suffix.lower() not in IMAGE_EXTENSIONS:
        raise RuntimeError(f"unsupported image extension {new_image_path.suffix!r}; accepted: {sorted(IMAGE_EXTENSIONS)}")
    new_image_origin = "user-supplied image (BYOD)"
else:
    new_image_path = NEW_IMAGE_DIR / "synthetic-new-warm.png"
    image = Image.new("RGB", (256, 256), (190, 65, 35))
    ImageDraw.Draw(image).ellipse((68, 68, 188, 188), outline=(255, 240, 220), width=12)
    image.save(new_image_path)
    new_image_origin = "synthetic new image drawn by this notebook (not in train/val)"

predicted, scores, _ = classify([new_image_path])
prediction = {
    "input": new_image_path.name,
    "inputOrigin": new_image_origin,
    "inputSha256": sha256_file(new_image_path),
    "decisionRule": "argmax(logits)",
    "predictedClass": class_names[int(predicted[0])],
    "uncalibratedSoftmaxScores": {name: float(scores[0][k]) for k, name in enumerate(class_names)},
    "classOrder": class_names,
}
print(json.dumps(prediction, indent=2))


## 10. Export machine-readable results and provenance

Four files are written to `WORK/exports/` (in Colab, open the folder icon on the left to download them):

| File | Contents |
|---|---|
| `validation-predictions.csv` | one row per validation sample: `sampleId`, `label`, `predicted`, one `score_<class>` column per class in `classOrder` |
| `metrics.json` | worker-reported metrics, majority baseline, and the fresh-boundary reload comparison |
| `prediction.json` | the new-image prediction from Section 9 |
| `provenance.json` | pipeline/worker/model revisions and digests, runtime, training configuration, dataset identity (`logicalDatasetDigest`), artifact bundle digest |

The deployable artifact itself is the generation directory under `training-output/artifact/generations/<generation>/` (`model.safetensors`, `model-config.json`, `artifact-manifest.json`). No secrets or tokens are written to any export.


In [ ]:
EXPORT_DIR = WORK / "exports"
EXPORT_DIR.mkdir(exist_ok=True)

with (EXPORT_DIR / "validation-predictions.csv").open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(per_sample[0].keys()))
    writer.writeheader()
    writer.writerows(per_sample)

metrics_export = {
    "notebookProfile": "E2E",
    "notebookSpec": "1.0",
    "reportedMetrics": reported_metrics,
    "comparison": comparison,
    "freshBoundaryReload": reload_check,
}
provenance_export = {
    "notebookProfile": "E2E",
    "notebookSpec": "1.0",
    "pipeline": {"repository": "kurtvalcorza/swin-classification-pipeline", "revision": PIPELINE_REF, "pipelineId": pipeline_manifest["pipelineId"], "taskProfile": pipeline_manifest["taskProfile"]},
    "workers": {
        "validator": {"revision": VALIDATOR_REF, "workerReleaseDigest": pipeline_manifest["validatorWorkerReleaseDigest"]},
        "finetuner": {"revision": FINETUNER_REF, "workerReleaseDigest": pipeline_manifest["finetunerWorkerReleaseDigest"]},
    },
    "baseModel": model_identity,
    "runtime": runtime,
    "dataset": {**dataset_origin, "logicalDatasetDigest": logical_manifest["logicalDatasetDigest"], "classNames": class_names, "splitCounts": split_counts},
    "training": {"method": "core.training.supervised-finetuning (full network, AdamW, cross-entropy)", "epochs": EPOCHS, "batchSize": BATCH_SIZE, "learningRate": LEARNING_RATE, "seed": SEED,
                 "reproducibility": run_manifest["reproducibility"], "device": run_manifest["observed"]["device"]},
    "artifact": {"generation": current_generation, "bundleDigest": artifact_manifest["bundleDigest"], "members": verified_members, "format": "safetensors + model-config.json + artifact-manifest.json"},
    "evaluation": comparison,
    "freshBoundaryReload": reload_check,
}
(EXPORT_DIR / "metrics.json").write_text(json.dumps(metrics_export, indent=2) + "\n")
(EXPORT_DIR / "prediction.json").write_text(json.dumps(prediction, indent=2) + "\n")
(EXPORT_DIR / "provenance.json").write_text(json.dumps(provenance_export, indent=2) + "\n")
for path in sorted(EXPORT_DIR.iterdir()):
    print(f"{path.stat().st_size:>8} bytes  {path}")


## Interpretation and limits

A successful top-to-bottom run **proves**, for the runtime printed in Section 1:

- the pipeline, validator and finetuner were checked out at the immutable revisions the release pins;
- the checkpoint bytes matched the allowlisted SHA-256 before any model was built;
- the pinned validator accepted the dataset and froze its identity;
- the pinned finetuner completed gradient fine-tuning on `cuda:0`, published a content-addressed artifact, and evaluated the *persisted* model;
- that artifact, copied to a fresh directory and rebuilt from its own contract, reproduced the worker's reported accuracy exactly and cross-entropy within tolerance;
- a new image could be classified from the reloaded artifact and every result was exported with provenance.

It does **not** prove: ImageNet-level or real-domain accuracy (the default data is synthetic and tiny; even BYOD metrics are a single holdout with no dispersion estimate), calibration of the softmax scores, robustness, fairness, safety, or production fitness. Formal accelerator qualification remains limited to the finetuner's recorded qualification packet; a clean run on Colab or Kaggle is execution evidence for *that* runtime only. Static notebook checks are not execution evidence — see `tutorials/RELEASE_VERIFICATION.md` for the recorded clean-runtime run of this notebook revision.

## Troubleshooting

| Symptom | Cause and fix |
|---|---|
| `RuntimeError: The validator and finetuner repositories are private…` | No `GITHUB_TOKEN` secret was found. Add it (Colab: key icon → Secrets, enable notebook access; Kaggle: Add-ons → Secrets) and rerun Section 1. |
| `git clone` fails with `403`/`Authentication failed` | The token exists but lacks read access to the two private repositories. |
| `CUDA is required…` or `ACCELERATOR_UNAVAILABLE` / `ACCELERATOR_MISMATCH` | Runtime has no `cuda:0`. Colab: Runtime → Change runtime type → GPU (T4), then Run all. |
| `timm … is active but 1.0.28 was installed` / Pillow WARNING in Section 1 | A package was imported before the pinned install (re-run in the same kernel, or a pre-warmed runtime). Runtime → Restart session, then run from the top. |
| `VISION_MISSING_VALIDATION_SPLIT` / `VISION_AMBIGUOUS_VALIDATION_SPLIT` | BYOD ZIP lacks `val/` (or has both `val/` and `valid/`). Fix the folder layout and rerun Sections 3–4. |
| `VISION_UNRECOGNIZED_IMAGE_EXTENSION` or a decode error | A non-image or corrupt file is inside a class folder. Remove it; nothing is skipped silently by design. |
| `CUDA out of memory` during Section 6 | Lower `BATCH_SIZE` (or choose the Tiny model) and rerun Section 6. |
| Checkpoint digest mismatch | Bytes served by the Hub differ from the catalog pin. Do not proceed; report it against the pipeline repository. |

## Next experiments

- Turn on `USE_BYOD_DATASET` with a small real image folder and compare the majority baseline with the model — this is the first result that says anything about your domain.
- Raise `EPOCHS` and watch `history`: on the synthetic sample validation loss collapses within an epoch; on real data look for the point where validation loss stops improving.
- Switch `MODEL_KEY` to the Small variant and compare cross-entropy at equal epochs and seed.
- Run twice with the same `SEED` and diff `provenance.json`: identical accuracy, slightly different cross-entropy and artifact digest — that is what *re-executable, not bitwise deterministic* means.
- Hand `fresh-reload/` to a separate process or machine and repeat Section 8 there; that is the boundary DIMER serving relies on.
